In [5]:
import pandas as pd

def clean_string(value):
    if pd.isna(value):
        return None
    return str(value).strip().lower()

def find_rate(rates_df, product, cover_type, age, tenure):
    matched = rates_df[
        (rates_df["product"].astype(str).str.strip().str.lower() == clean_string(product)) &
        (rates_df["cover_type"].astype(str).str.strip().str.lower() == clean_string(cover_type)) &
        (rates_df["age_min"] <= age) &
        (rates_df["age_max"] >= age) &
        (rates_df["tenure_months"] == tenure)
    ]

    if matched.empty:
        return None

    return matched.iloc[0]["rate_per_lakh"]

def calculate_premium_row(row, rates_df, selected_product, selected_cover):
    try:
        age = int(row["Age"])
        tenure = int(row["Tenure_Months"])
        loan_amount = float(row["Loan_Amount"])

        rate = find_rate(rates_df, selected_product, selected_cover, age, tenure)

        if rate is None:
            return pd.Series({
                "Selected_Product": selected_product,
                "Selected_Cover_Type": selected_cover,
                "Rate_Per_Lakh": None,
                "Premium": None,
                "Status": "Rate not found"
            })

        premium = (loan_amount / 100000) * rate

        return pd.Series({
            "Selected_Product": selected_product,
            "Selected_Cover_Type": selected_cover,
            "Rate_Per_Lakh": round(rate, 2),
            "Premium": round(premium, 2),
            "Status": "Success"
        })

    except Exception as e:
        return pd.Series({
            "Selected_Product": selected_product,
            "Selected_Cover_Type": selected_cover,
            "Rate_Per_Lakh": None,
            "Premium": None,
            "Status": f"Error: {str(e)}"
        })

In [6]:
sample_customer_df = pd.DataFrame({
    "Customer_ID": [101, 102, 103, 104],
    "Customer_Name": ["Amit", "Neha", "Ravi", "Sneha"],
    "Age": [18, 25, 35, 40],
    "Tenure_Months": [12, 24, 36, 60],
    "Loan_Amount": [500000, 750000, 300000, 1000000]
})

sample_customer_df.to_excel("customer_data.xlsx", index=False)
sample_customer_df

,Customer_ID,Customer_Name,Age,Tenure_Months,Loan_Amount
0,101,Amit,18,12,500000
1,102,Neha,25,24,750000
2,103,Ravi,35,36,300000
3,104,Sneha,40,60,1000000


In [10]:
rates_df = pd.read_csv("normalized_rates.csv")
customer_df = pd.read_excel("customer_data.xlsx")

selected_product = "Unsecured Business Loan"
selected_cover = "Level Cover"

result_df = customer_df.copy()

calc_output = result_df.apply(
    lambda row: calculate_premium_row(
        row=row,
        rates_df=rates_df,
        selected_product=selected_product,
        selected_cover=selected_cover
    ),
    axis=1
)

result_df = pd.concat([result_df, calc_output], axis=1)
result_df

,Customer_ID,Customer_Name,Age,Tenure_Months,Loan_Amount,Selected_Product,Selected_Cover_Type,Rate_Per_Lakh,Premium,Status
0,101,Amit,18,12,500000,Unsecured Business Loan,Level Cover,362.85,1814.25,Success
1,102,Neha,25,24,750000,Unsecured Business Loan,Level Cover,710.94,5332.05,Success
2,103,Ravi,35,36,300000,Unsecured Business Loan,Level Cover,1367.76,4103.28,Success
3,104,Sneha,40,60,1000000,Unsecured Business Loan,Level Cover,3243.51,32435.10,Success


In [12]:
result_df.to_excel("premium_output2.xlsx", index=False)
result_df.to_csv("premium_output2.csv", index=False)

print("Premium output file created successfully.")

Premium output file created successfully.


In [13]:
#input validation
def validate_customer_file(df):
    required_cols = ["Customer_ID", "Customer_Name", "Age", "Tenure_Months", "Loan_Amount"]
    missing_cols = [col for col in required_cols if col not in df.columns]

    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

    return True

In [14]:
validate_customer_file(customer_df)
print("Customer file format is valid.")

Customer file format is valid.


In [15]:
print("Products:")
print(sorted(rates_df["product"].dropna().unique().tolist()))

print("\nCover Types:")
print(sorted(rates_df["cover_type"].dropna().unique().tolist()))

Products:
['Home Loan', 'Loan Against Property', 'Micro Loan', 'Personal Loan', 'Secured Loan Business', 'Unsecured Business Loan']

Cover Types:
['Level Cover', 'Reducing']
